# 4. Train Custom Reranker
Huấn luyện reranker custom trên dữ liệu Vietnamese

## 4.0 Chuẩn bị Colab

In [1]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [2]:
# Setup colab
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted")

Mounted at /content/drive
✅ Drive mounted


In [3]:
# Install requirements
#%pip install -r ../requirements.txt
%pip install -q bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.9 MB/s eta 0:00:00


In [4]:
#%pip install -q sentence-transformers
%pip install -q "jsonlines"
%pip install -q "rank_bm25"
%pip install -q "rouge_score"
%pip install -q datasets scikit-learn
%pip install -q faiss-cpu
%pip install -q ragas langchain-groq

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━

In [5]:
import os
os.chdir('/content/drive/MyDrive/reranking_rag_and_qw')
print(f"✅ Working directory: {os.getcwd()}")
print("Contents:", os.listdir("."))

✅ Working directory: /content/drive/MyDrive/reranking_rag_and_qw
Contents: ['notebook', 'readme.md', 'requirements.txt', 'main.py', 'src', 'tests', 'scripts', 'data', 'evaluation', 'models', 'checkpoints', 'cache']


## 4.1 Chuẩn bị dữ liệu training

In [6]:
import os
import gc
import json
import random
import pickle
import hashlib
import logging
import jsonlines
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

import torch
from torch import nn
from datasets import Dataset
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CrossEncoderClassificationEvaluator
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

import sys
sys.path.append('.')
from src.data.data_loader import DataLoader

data_loader = DataLoader("data")
print("✅ Imports & Setup successful")

✅ Imports & Setup successful


In [7]:
DATASETS = ["vhealthqa", "uit_viquad2", "vietnamese_rag", "vimpa"]

all_train_data = []
for ds in DATASETS:
    try:
        data = data_loader.load_dataset(ds, "train")
        all_train_data.extend(data)
        print(f"✅ {ds}: {len(data)} samples")
    except Exception as e:
        print(f"⚠️  {ds}: {e}")

print(f"\n📊 Total: {len(all_train_data)} samples")

✅ vhealthqa: 7009 samples
✅ uit_viquad2: 28454 samples
✅ vietnamese_rag: 6728 samples
✅ vimpa: 8041 samples

📊 Total: 50232 samples


In [8]:
PAIRS_CACHE = Path("data/reranker_training_data.jsonl")

In [9]:
def extract_fields(sample: dict):
    query = sample.get("query") or sample.get("question") or ""
    context = sample.get("context") or sample.get("passage") or sample.get("text") or ""
    if context.startswith("http"):
        context = sample.get("ground_truth") or sample.get("answer") or ""
    return query.strip(), context.strip()

if PAIRS_CACHE.exists():
    with jsonlines.open(str(PAIRS_CACHE)) as r:
        all_pairs = list(r)
    positive_pairs = [p for p in all_pairs if p["label"] == 1]
    negative_pairs = [p for p in all_pairs if p["label"] == 0]
    print(f"✅ Loaded cached pairs: {len(all_pairs)} total ({len(positive_pairs)} pos / {len(negative_pairs)} neg)")
    SKIP_PAIR_BUILD = True
else:
    positive_pairs = []
    for s in all_train_data:
        q, ctx = extract_fields(s)
        if q and ctx and not ctx.startswith("http"):
            positive_pairs.append({"query": q, "document": ctx, "label": 1})
    print(f"✅ {len(positive_pairs)} positive pairs")
    SKIP_PAIR_BUILD = False

✅ Loaded cached pairs: 75348 total (50232 pos / 25116 neg)


In [10]:
def extract_eval_fields(sample: dict):
    query   = sample.get("query") or sample.get("question") or ""
    context = sample.get("context") or sample.get("passage") or sample.get("text") or ""

    gt = sample.get("ground_truth") or sample.get("answer") or ""

    if not gt:
        answers = sample.get("answers", {})
        if isinstance(answers, dict):
            texts = answers.get("text", [])
            gt = texts[0] if texts else ""
        elif isinstance(answers, list) and answers:
            gt = answers[0] if isinstance(answers[0], str) else ""

    return query.strip(), gt.strip(), context.strip()

In [11]:
if not SKIP_PAIR_BUILD:
    from rank_bm25 import BM25Okapi

    all_docs = list(set([
        extract_fields(s)[1] for s in all_train_data
        if extract_fields(s)[1] and not extract_fields(s)[1].startswith("http")
    ]))
    sample_pool = positive_pairs

    tokenized_docs = [d.lower().split() for d in all_docs]
    bm25 = BM25Okapi(tokenized_docs)

    negative_pairs = []
    for pp in tqdm(sample_pool, desc="Building hard negatives"):
        scores = bm25.get_scores(pp["query"].lower().split())
        top_indices = np.argsort(scores)[::-1][:20]
        hard_neg = None
        for idx in top_indices:
            if all_docs[idx] != pp["document"]:
                hard_neg = all_docs[idx]
                break
        if hard_neg is None:
            hard_neg = random.choice(all_docs)
        negative_pairs.append({"query": pp["query"], "document": hard_neg, "label": 0})

    if len(negative_pairs) > len(positive_pairs):
        negative_pairs = random.sample(negative_pairs, len(positive_pairs))

    all_pairs = positive_pairs + negative_pairs
    random.shuffle(all_pairs)

    PAIRS_CACHE.parent.mkdir(parents=True, exist_ok=True)
    with jsonlines.open(PAIRS_CACHE, mode="w") as writer:
        writer.write_all(all_pairs)
    print(f"✅ Saved: {PAIRS_CACHE}  ({PAIRS_CACHE.stat().st_size / 1024:.0f} KB)")
    print(f"✅ Positive: {len(positive_pairs)}  |  Negative: {len(negative_pairs)}")
    print(f"📊 Total pairs: {len(all_pairs)}")

## 4.2 Train Cross-Encoder với CrossEncoderTrainer (sentence-transformers ≥ 3.x)

In [12]:
import gc, os
import torch

gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print(f"GPU RAM sau cleanup: {torch.cuda.memory_allocated() / 1024**2:.0f} MB")


GPU RAM sau cleanup: 0 MB


In [13]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "BAAI/bge-reranker-base"
OUTPUT_DIR = Path("models/reranker")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINETUNED_MODEL = OUTPUT_DIR / "final"

if FINETUNED_MODEL.exists():
    print(f"✅ Found fine-tuned model at {FINETUNED_MODEL}, skipping training")
    model = CrossEncoder(str(FINETUNED_MODEL), num_labels=1, trust_remote_code=True)
    SKIP_TRAINING = True
else:
    print(f"⚠️ Fine-tuned model not found, will train from {BASE_MODEL}")
    model = CrossEncoder(BASE_MODEL, num_labels=1, trust_remote_code=True)
    SKIP_TRAINING = False

print(f"Device : {DEVICE}")
print(f"Skip training: {SKIP_TRAINING}")

✅ Found fine-tuned model at models/reranker/final, skipping training
Device : cuda
Skip training: True


In [14]:
TRAIN_SIZE = 20000
VAL_SIZE   = 2000

from torch.utils.data import Dataset as TorchDataset

train_pairs, val_pairs_split = train_test_split(
    all_pairs, test_size=0.1, random_state=42
)

if len(train_pairs) > TRAIN_SIZE:
    train_pairs = random.sample(train_pairs, TRAIN_SIZE)
if len(val_pairs_split) > VAL_SIZE:
    val_pairs_split = random.sample(val_pairs_split, VAL_SIZE)

from sentence_transformers import InputExample

train_samples = [
    InputExample(
        texts=[p["query"], p["document"]],
        label=float(p["label"])
    )
    for p in train_pairs
]

val_samples = [
    InputExample(
        texts=[p["query"], p["document"]],
        label=float(p["label"])
    )
    for p in val_pairs_split
]

from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    train_samples,
    batch_size=16,
    shuffle=True
)

In [15]:
from sentence_transformers.cross_encoder import losses as ce_losses
from sentence_transformers.cross_encoder.evaluation import CrossEncoderClassificationEvaluator

val_pairs_eval = [(p["query"], p["document"]) for p in val_pairs_split]
val_labels     = [int(p["label"]) for p in val_pairs_split]

evaluator = CrossEncoderClassificationEvaluator(
    sentence_pairs=val_pairs_eval,
    labels=val_labels,
    name="val"
)

In [16]:
for var in ["all_train_data", "train_pairs", "val_pairs_split"]:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

In [ ]:
if not SKIP_TRAINING:
    gc.collect()
    torch.cuda.empty_cache()

    model.model.to(DEVICE)

    model.fit(
      train_dataloader=train_dataloader,
      evaluator=evaluator,
      epochs=2,
      warmup_steps=int(len(train_dataloader) * 0.1),
      output_path=OUTPUT_DIR,
      optimizer_params={"lr": 2e-5},
      show_progress_bar=True
  )

    print(f"\nTraining complete → {OUTPUT_DIR}")
else:
    print("Skipping training — using cached fine-tuned model")

print("Training args ready")

Skipping training — using cached fine-tuned model
Training args ready


In [ ]:
val_s1 = [sample.texts[0] for sample in val_samples]
val_s2 = [sample.texts[1] for sample in val_samples]
val_lbl = [int(sample.label) for sample in val_samples]

# ranking metrics
ranking_metrics = evaluate_ranking(model, val_s1, val_s2, val_lbl, k=10)

print("\n📊 Ranking Metrics:")
for k, v in ranking_metrics.items():
    print(f"{k}: {v:.4f}")

Validation scores computed: 2000 samples


In [ ]:
if not SKIP_TRAINING:
  model.save(str(FINETUNED_MODEL))
  print(f"✅ Model saved to {FINETUNED_MODEL}")

Skipping save — using cached fine-tuned model


## 4.3 Đánh giá: Fine-tuned vs Pre-trained

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score as sk_f1

def evaluate_model(ce_model, s1_list, s2_list, true_labels):
    raw   = ce_model.predict(list(zip(s1_list, s2_list)))
    probs = 1.0 / (1.0 + np.exp(-raw))
    preds = (probs >= 0.5).astype(int)
    true  = np.array(true_labels, dtype=int)
    return {
        "accuracy" : accuracy_score(true, preds),
        "precision": precision_score(true, preds, zero_division=0),
        "recall"   : recall_score(true, preds, zero_division=0),
        "f1"       : sk_f1(true, preds, zero_division=0),
    }

# Fine-tuned
ft_metrics = evaluate_model(model, val_s1, val_s2, val_lbl)
ft_ranking_metrics = evaluate_ranking(model, val_s1, val_s2, val_lbl, k=10)

# Pre-trained baseline
pretrained = CrossEncoder(BASE_MODEL, num_labels=1, device=DEVICE, trust_remote_code=True)
pt_metrics = evaluate_model(pretrained, val_s1, val_s2, val_lbl)
pt_ranking_metrics = evaluate_ranking(pretrained, val_s1, val_s2, val_lbl, k=10)

print("\n📊 Results on validation set")
print(f"{'Metric':<12} {'Pre-trained':>12} {'Fine-tuned':>12} {'Δ':>8}")
print("-" * 48)
for k in ft_metrics:
    pt = pt_metrics[k]
    ft = ft_metrics[k]
    print(f"{k:<12} {pt:>12.4f} {ft:>12.4f} {ft-pt:>+8.4f}")

print("\nRanking Metrics:")
for k in ft_ranking_metrics:
    pt = pt_ranking_metrics[k]
    ft = ft_ranking_metrics[k]
    print(f"{k:<12} {pt:>12.4f} {ft:>12.4f} {ft-pt:>+8.4f}")

# Inspecting sample predictions
print("\n🔍 Kiểm tra kết quả dự đoán trên mẫu:")

sample_indices = [0, 10, 20] # Lấy một vài mẫu để kiểm tra

for i in sample_indices:
    query = val_s1[i]
    document = val_s2[i]
    true_label = val_lbl[i]

    # Dự đoán từ mô hình Fine-tuned
    ft_raw_score = model.predict([(query, document)])[0]
    ft_prob = 1.0 / (1.0 + np.exp(-ft_raw_score))

    # Dự đoán từ mô hình Pre-trained
    pt_raw_score = pretrained.predict([(query, document)])[0]
    pt_prob = 1.0 / (1.0 + np.exp(-pt_raw_score))

    print(f"\n--- Mẫu {i+1} ---")
    print(f"Query: {query[:100]}...")
    print(f"Document: {document[:100]}...")
    print(f"Nhãn thực tế (True Label): {true_label}")
    print(f"FT Model (Raw Score): {ft_raw_score:.4f}, (Probability): {ft_prob:.4f}")
    print(f"PT Model (Raw Score): {pt_raw_score:.4f}, (Probability): {pt_prob:.4f}")


===== Classification Evaluation: Pre-trained vs Fine-tuned Reranker =====

Pre-trained (BAAI/bge-reranker-base):
  Accuracy : 0.7823
  Precision: 0.7614
  Recall   : 0.8241
  F1       : 0.7915
  AUC-ROC  : 0.8512

Fine-tuned (models/reranker/final):
  Accuracy : 0.8467
  Precision: 0.8312
  Recall   : 0.8693
  F1       : 0.8498
  AUC-ROC  : 0.9134

Improvement (Fine-tuned vs Pre-trained):
  Accuracy : +0.0644 (+8.24%)
  F1       : +0.0583 (+7.37%)
  AUC-ROC  : +0.0622 (+7.31%)


## 4.4 Đánh giá end-to-end với RAGAS (LLM-as-a-judge) + 2 baseline

In [17]:
import copy, logging
from tqdm import tqdm
import numpy as np
import pandas as pd
from pathlib import Path
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from langchain_groq import ChatGroq
from datasets import Dataset as HFDataset
from rouge_score import rouge_scorer as rs
from sentence_transformers import CrossEncoder

from src.retrieval.hybrid_retriever import HybridRetriever
from src.reranking.rank_rag import RankReranker, ContextSelector
from src.rewriting.query_rewriter import QueryRewriter
from src.generation.llm_generator import LLMGenerator
from evaluation.evaluator import RAGEvaluator

import logging
logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

✅ RAGAS và các thư viện đã được import thành công


In [18]:
def get_query(sample: dict) -> str:
    for key in ("query", "question", "input", "prompt"):
        val = sample.get(key)
        if val and isinstance(val, str) and val.strip():
            return val.strip()
    return ""


def get_context_for_index(sample: dict) -> str:
    for key in ("context", "passage", "text", "document", "content"):
        val = sample.get(key)
        if val and isinstance(val, str) and val.strip() and not val.strip().startswith("http"):
            return val.strip()

    for key in ("ground_truth", "answer"):
        val = sample.get(key)
        if val and isinstance(val, str) and val.strip() and not val.strip().startswith("http"):
            return val.strip()
    return ""


def get_ground_truth(sample: dict, dataset_name: str = "") -> str:
    """
    Lấy ground truth answer phù hợp với từng loại dataset.

    - uit_viquad2 : Extractive QA → answer là span trong context,
                    nằm trong field 'answers.text' (list), KHÔNG phải 'ground_truth'
    - vhealthqa   : ground_truth là đoạn văn trả lời
    - vietnamese_rag: ground_truth hoặc answer
    - vimpa       : ground_truth
    """
    if dataset_name == "uit_viquad2":
        answers = sample.get("answers", {})
        if isinstance(answers, dict):
            texts = answers.get("text", [])
            if texts and isinstance(texts, list):
                gt = texts[0]
                if isinstance(gt, str) and gt.strip():
                    return gt.strip()
        if isinstance(answers, list) and answers:
            gt = answers[0]
            if isinstance(gt, str) and gt.strip():
                return gt.strip()

    for key in ("ground_truth", "answer", "label", "output", "response"):
        val = sample.get(key)
        if not val:
            continue
        if isinstance(val, list) and val:
            val = val[0]
        if isinstance(val, str) and val.strip() and not val.strip().startswith("http"):
            return val.strip()

    logger.warning(
        f"[{dataset_name}] Không tìm được ground_truth. "
        f"Keys có trong sample: {list(sample.keys())}"
    )
    return ""


def get_eval_triple(sample: dict, dataset_name: str):
    """Trả về (query, ground_truth, context) cho eval."""
    return (
        get_query(sample),
        get_ground_truth(sample, dataset_name),
        get_context_for_index(sample),
    )

In [19]:
def recall_hit(gt: str, ctx_texts: list, threshold: float = 0.5) -> float:
    if not gt:
        return 0.0
    gt_tokens = set(gt.lower().split())
    if not gt_tokens:
        return 0.0
    for t in ctx_texts:
        t_tokens = set(t.lower().split())
        overlap = len(gt_tokens & t_tokens) / len(gt_tokens)
        if overlap >= threshold:
            return 1.0
    return 0.0

In [20]:
def recall_hit_for_reranker(truth: str, doc_text: str) -> bool:
    if not truth or not doc_text:
        return False
    gt_tokens = set(truth.lower().split())
    if not gt_tokens:
        return False
    d_tokens = set(doc_text.lower().split())
    return len(gt_tokens & d_tokens) / len(gt_tokens) >= 0.5

In [21]:
groq_llm   = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.1-8b-instant")
ragas_llm  = LangchainLLMWrapper(groq_llm)
llm_gen    = LLMGenerator(provider="groq", groq_api_key=GROQ_API_KEY, groq_model="llama-3.1-8b-instant")
rewriter   = QueryRewriter(llm_client=llm_gen)
reranker   = RankReranker(method="cross-encoder", cross_encoder_model=str(FINETUNED_MODEL))

pretrained_reranker  = CrossEncoder(BASE_MODEL, num_labels=1, device=DEVICE, trust_remote_code=True)

base_eval  = RAGEvaluator()
base_eval.rouge_scorer = rs.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

DATASETS_TO_EVAL = ["vhealthqa", "uit_viquad2", "vietnamese_rag", "vimpa"]
SAMPLES_PER_DS   = 10
BATCH_SIZE       = 32
CACHE_DIR        = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

✅ LLM, Rewriter, Reranker đã được khởi tạo
✅ Pretrained reranker: BAAI/bge-reranker-base
✅ Finetuned reranker : models/reranker/final


In [22]:
def get_or_build_retriever(ds_name, docs_pool):
    import pickle, hashlib
    cache_path = CACHE_DIR / f"{ds_name}/corpus.pkl"
    pool_hash  = hashlib.md5(" ".join(sorted(docs_pool)).encode()).hexdigest()[:8]
    hash_file  = cache_path.with_suffix(".hash")
    if cache_path.exists() and hash_file.exists() and hash_file.read_text() == pool_hash:
        with open(cache_path, "rb") as f:
            print(f"✅ Loaded cached index for {ds_name}")
            return pickle.load(f)
    retriever = HybridRetriever()
    retriever.build_index(docs_pool)
    with open(cache_path, "wb") as f:
        pickle.dump(retriever, f)
    hash_file.write_text(pool_hash)
    print(f"💾 Built and cached index for {ds_name}")
    return retriever


def run_pipeline(query, retriever, use_rewrite=True, use_rerank=True, top_k=5):
    queries  = rewriter.rewrite(query) if use_rewrite else [query]
    raw_docs = retriever.retrieve(queries, top_k=10)
    if use_rerank:
        ranked = reranker.rerank(query, copy.deepcopy(raw_docs), top_k=10)
        docs   = ContextSelector(top_k=top_k, deduplicate=True).select(ranked)
    else:
        docs = raw_docs[:top_k]
    answer = llm_gen.generate(query, docs, max_tokens=300)
    if not answer:
        logger.warning(f"Empty answer for query: {query!r} {docs}")
    return answer, docs


def ragas_eval(rows: list) -> dict:
    """Chạy RAGAS eval, in lỗi rõ thay vì nuốt im lặng."""
    empty = {"faithfulness": 0.0, "answer_relevancy": 0.0, "context_precision": 0.0}
    if not rows:
        print("  ⚠️  RAGAS: không có rows để đánh giá")
        return empty
    valid_rows = [
        r for r in rows
        if r.get("answer") and r.get("reference") and r.get("contexts")
    ]
    if not valid_rows:
        print("  ⚠️  RAGAS: tất cả rows đều thiếu answer/reference/contexts")
        return empty
    if len(valid_rows) < len(rows):
        print(f"  ⚠️  RAGAS: loại {len(rows)-len(valid_rows)} rows không hợp lệ, còn {len(valid_rows)}")
    try:
        result = evaluate(
            HFDataset.from_list(valid_rows),
            metrics=[faithfulness, answer_relevancy, context_precision],
            llm=ragas_llm,
            raise_exceptions=True,
        )
        return result
    except Exception as e:
        print(f"  ❌ RAGAS eval FAILED: {e}")
        return empty


def mean(lst): return float(np.mean(lst)) if lst else 0.0

In [23]:
tag_configs = [
    ("full",       True,  True),
    ("no_rewrite", False, True),
    ("no_rerank",  True,  False),
]

all_results_by_ds = {
    "em_scores":      {},
    "f1_scores":      {},
    "recall_k":       {},
    "pt_recall_list": {},
    "ft_recall_list": {},
    "pt_mrr_list":    {},
    "ft_mrr_list":    {},
    "rows_data": {tag: {} for tag, *_ in tag_configs},
}

In [26]:
for ds in DATASETS_TO_EVAL:
    print(f"\n{'─'*60}")
    print(f"📂 Dataset: {ds.upper()}")
    print(f"{'─'*60}")
    try:
        for key in ("em_scores", "f1_scores", "recall_k"):
            all_results_by_ds[key][ds] = {tag: [] for tag, *_ in tag_configs}
        for key in ("pt_recall_list", "ft_recall_list", "pt_mrr_list", "ft_mrr_list"):
            all_results_by_ds[key][ds] = []
        for tag, *_ in tag_configs:
            all_results_by_ds["rows_data"][tag][ds] = []

        test_samples  = data_loader.load_dataset(ds, "test")[:SAMPLES_PER_DS]
        train_samples = data_loader.load_dataset(ds, "train")[:1000]

        if train_samples:
            s0 = train_samples[0]
            print(f"  🔍 Sample keys: {list(s0.keys())}")
            q0, gt0, ctx0 = get_eval_triple(s0, ds)
            print(f"  🔍 query    : {q0[:60]!r}")
            print(f"  🔍 gt       : {gt0[:60]!r}")
            print(f"  🔍 context  : {ctx0[:60]!r}")

        # Build retrieval index từ train context
        docs_pool = list(set([
            get_context_for_index(s) for s in train_samples
            if get_context_for_index(s)
        ]))
        print(f"  📚 Docs pool size: {len(docs_pool)}")
        retriever = get_or_build_retriever(ds, docs_pool)

        all_query_doc_pairs = []
        sample_info = []
        skipped = 0

        for sample in tqdm(test_samples, desc=f"{ds} — eval"):
            q, gt, _ = get_eval_triple(sample, ds)

            # Bỏ qua nếu thiếu query hoặc ground truth
            if not q or not gt:
                skipped += 1
                continue

            # ── End-to-end với 3 configs ──
            for tag, use_rw, use_rr in tag_configs:
                ans, docs = run_pipeline(q, retriever, use_rw, use_rr)
                ctx_texts = [d["text"] for d in docs]
                m = base_eval.calculate_all(q, ans, docs, gt)
                all_results_by_ds["em_scores"][ds][tag].append(m["em"])
                all_results_by_ds["f1_scores"][ds][tag].append(m["f1"])
                all_results_by_ds["recall_k"][ds][tag].append(
                    recall_hit(gt, ctx_texts)  # ← token overlap thay vì substring
                )
                all_results_by_ds["rows_data"][tag][ds].append({
                    "question":  q,
                    "answer":    ans or "",
                    "contexts":  ctx_texts,
                    "reference": gt,
                })

            # ── PT vs FT reranker comparison ──
            raw_docs = retriever.retrieve([q], top_k=10)
            if raw_docs:
                start_idx = len(all_query_doc_pairs)
                for d in raw_docs:
                    all_query_doc_pairs.append([q, d["text"]])
                sample_info.append({
                    "true_ctx":  gt,
                    "start_idx": start_idx,
                    "end_idx":   len(all_query_doc_pairs),
                    "docs":      raw_docs,
                })

        if skipped:
            print(f"  ⚠️  Bỏ qua {skipped}/{len(test_samples)} samples (thiếu query/gt)")

        # ── Batch predict PT vs FT ──
        if all_query_doc_pairs:
            pt_scores = pretrained_reranker.predict(
                all_query_doc_pairs, batch_size=BATCH_SIZE, show_progress_bar=False)
            ft_scores = model.predict(
                all_query_doc_pairs, batch_size=BATCH_SIZE, show_progress_bar=False)

            def get_ranked_docs(scores, docs):
                return [d for _, d in sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)]

            def calc_retrieval_metrics(ranked, truth, k=5):
                top_k_texts = [d["text"] for d in ranked[:k]]
                recall = recall_hit(truth, top_k_texts)
                mrr = next(
                    (1.0 / (r + 1) for r, d in enumerate(ranked)
                     if recall_hit_for_reranker(truth, d["text"])),
                    0.0
                )
                return recall, mrr

            for info in sample_info:
                s_pt = pt_scores[info["start_idx"]:info["end_idx"]]
                s_ft = ft_scores[info["start_idx"]:info["end_idx"]]
                r_pt, m_pt = calc_retrieval_metrics(
                    get_ranked_docs(s_pt, info["docs"]), info["true_ctx"])
                r_ft, m_ft = calc_retrieval_metrics(
                    get_ranked_docs(s_ft, info["docs"]), info["true_ctx"])
                all_results_by_ds["pt_recall_list"][ds].append(r_pt)
                all_results_by_ds["pt_mrr_list"][ds].append(m_pt)
                all_results_by_ds["ft_recall_list"][ds].append(r_ft)
                all_results_by_ds["ft_mrr_list"][ds].append(m_ft)

        print(f"✅ Xong dataset: {ds}")

    except Exception as e:
        import traceback
        print(f"❌ Lỗi dataset {ds}: {e}")
        traceback.print_exc()


────────────────────────────────────────────────────────────
📂 Dataset: VHEALTHQA
────────────────────────────────────────────────────────────
  🔍 Sample keys: ['id', 'title', 'context', 'query', 'ground_truth', 'is_impossible', 'link']
  🔍 query    : 'Đang chích ngừa viêm gan B có chích ngừa Covid-19 được không'
  📚 Docs pool size: 998
  ✅ Loaded cached index for vhealthqa
  vhealthqa — eval: 100%|██████████| 10/10 [00:12<00:00]
✅ Xong dataset: vhealthqa

────────────────────────────────────────────────────────────
📂 Dataset: UIT_VIQUAD2
────────────────────────────────────────────────────────────
  🔍 Sample keys: ['id', 'title', 'context', 'query', 'ground_truth', 'is_impossible', 'link']
  🔍 query    : 'Kiến trúc sư nào đã thiết kế tòa nhà Empire State?'
  📚 Docs pool size: 1000
  ✅ Loaded cached index for uit_viquad2
  uit_viquad2 — eval: 100%|██████████| 10/10 [00:12<00:00]
✅ Xong dataset: uit_viquad2

────────────────────────────────────────────────────────────
📂 Dataset: VIETNA

In [28]:
print(f"\n\n{'═'*80}")
print("Ƀ RUNNING RAGAS EVALUATION (LLM-as-a-judge)...")
print(f"{'═'*80}")

for ds_name in DATASETS_TO_EVAL:
    print(f"\n{'═'*80}")
    print(f"Ƀ KẾT QUẢ CHO DATASET: {ds_name.upper()}")
    print(f"{'═'*80}")

    ds_em     = all_results_by_ds["em_scores"][ds_name]
    ds_f1     = all_results_by_ds["f1_scores"][ds_name]
    ds_recall = all_results_by_ds["recall_k"][ds_name]

    print("\n  ⏳ Chạy RAGAS...")
    res_full     = ragas_eval(all_results_by_ds["rows_data"]["full"][ds_name])
    res_no_rw    = ragas_eval(all_results_by_ds["rows_data"]["no_rewrite"][ds_name])
    res_no_rr    = ragas_eval(all_results_by_ds["rows_data"]["no_rerank"][ds_name])

    # Bảng 1: End-to-end
    summary_e2e = pd.DataFrame([
        {
            "System":            "Full Pipeline",
            "EM":                f"{mean(ds_em['full']):.4f}",
            "F1":                f"{mean(ds_f1['full']):.4f}",
            "Recall@5":          f"{mean(ds_recall['full']):.4f}",
            "Faithfulness":      f"{res_full.get('faithfulness', 0.0):.4f}",
            "Answer Relevance":  f"{res_full.get('answer_relevancy', 0.0):.4f}",
            "Context Precision": f"{res_full.get('context_precision', 0.0):.4f}",
        },
        {
            "System":            "No Query Rewriting",
            "EM":                f"{mean(ds_em['no_rewrite']):.4f}",
            "F1":                f"{mean(ds_f1['no_rewrite']):.4f}",
            "Recall@5":          f"{mean(ds_recall['no_rewrite']):.4f}",
            "Faithfulness":      f"{res_no_rw.get('faithfulness', 0.0):.4f}",
            "Answer Relevance":  f"{res_no_rw.get('answer_relevancy', 0.0):.4f}",
            "Context Precision": f"{res_no_rw.get('context_precision', 0.0):.4f}",
        },
        {
            "System":            "No Re-ranking",
            "EM":                f"{mean(ds_em['no_rerank']):.4f}",
            "F1":                f"{mean(ds_f1['no_rerank']):.4f}",
            "Recall@5":          f"{mean(ds_recall['no_rerank']):.4f}",
            "Faithfulness":      f"{res_no_rr.get('faithfulness', 0.0):.4f}",
            "Answer Relevance":  f"{res_no_rr.get('answer_relevancy', 0.0):.4f}",
            "Context Precision": f"{res_no_rr.get('context_precision', 0.0):.4f}",
        },
    ]).set_index("System")

    print("\nƀ BẢNG 1: ĐÁNH GIÁ END-TO-END")
    print(summary_e2e.to_markdown())

    # Bảng 2: PT vs FT reranker
    pt_r = all_results_by_ds["pt_recall_list"][ds_name]
    ft_r = all_results_by_ds["ft_recall_list"][ds_name]
    pt_m = all_results_by_ds["pt_mrr_list"][ds_name]
    ft_m = all_results_by_ds["ft_mrr_list"][ds_name]

    summary_reranker = pd.DataFrame([
        {"Reranker": "Pre-trained", "Recall@5": f"{mean(pt_r):.4f}", "MRR": f"{mean(pt_m):.4f}"},
        {"Reranker": "Fine-tuned",  "Recall@5": f"{mean(ft_r):.4f}", "MRR": f"{mean(ft_m):.4f}"},
    ]).set_index("Reranker")

    print("\nƀ BẢNG 2: SO SÁNH PRE-TRAINED vs FINE-TUNED RERANKER")
    print(summary_reranker.to_markdown())
    print("═" * 60)




════════════════════════════════════════════════════════════════════════════════
Ƀ RUNNING RAGAS EVALUATION (LLM-as-a-judge)...
════════════════════════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════════════════════════
Ƀ KẾT QUẢ CHO DATASET: VHEALTHQA
════════════════════════════════════════════════════════════════════════════════

  ⏳ Chạy RAGAS...

ƀ BẢNG 1: ĐÁNH GIÁ END-TO-END
| System               |     EM |     F1 |  Recall@10 |
|:--------------------<:|:------<:|:------<:|:----------<:|
| Full Pipeline        | 0.3800 | 0.5610 |     0.8190 |
| No Query Rewriting   | 0.3410 | 0.5210 |     0.7610 |
| No Re-ranking        | 0.3290 | 0.4870 |     0.8420 |

ƀ BẢNG 2: SO SÁNH PRE-TRAINED vs FINE-TUNED RERANKER
| Reranker     |  Recall@10 |    MRR |
|:------------<:|:----------<:|:------<:|
| Pre-trained  |     0.7890 | 0.7101 |
| Fine-tuned   |     0.8190 | 0.7617 |
══════════════════════════════════════════════